Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# これまでの進捗（簡易版）

| Ver. | 内容 | CV スコア | LB スコア |
| ---- | ---- | ---- | ---- |
| 1 | LightGBM によるパイプライン構築 | 0.95841 | 0.95815 |
| 2 | 周波数エンコーディング | 0.95910 | 0.95877 |
| 3 | EDA（travel_customer_type） | 0.95920 | 0.95897 |

# データの読み込み・ライブラリのインポート

In [73]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import kagglehub
import os

In [74]:
path = kagglehub.competition_download("playground-series-s6e10")

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))

In [75]:
train.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


In [76]:
test.head()

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Leg room service,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class
0,699635,25,1197,4,4,4,2,3,4,3,...,5,2,1,3,0,0.0,Female,disloyal Customer,Business travel,Eco
1,699636,36,829,5,5,5,5,5,4,5,...,2,4,3,5,0,0.0,Male,Loyal Customer,Business travel,Eco
2,699637,45,472,4,5,5,5,4,4,4,...,3,1,3,4,0,0.0,Male,Loyal Customer,Business travel,Eco
3,699638,45,762,2,2,2,5,3,2,3,...,4,4,4,3,0,0.0,Female,disloyal Customer,Business travel,Business
4,699639,34,594,1,2,1,1,2,1,2,...,4,2,1,2,0,0.0,Male,Loyal Customer,Personal Travel,Eco Plus


In [77]:
sample_submission.head()

,id,satisfaction
0,699635,0.443573
1,699636,0.443573
2,699637,0.443573
3,699638,0.443573
4,699639,0.443573


In [78]:
print(f"train の件数 = {len(train)}")
print(f"test の件数 = {len(test)}")
print(f"sample_submission の件数 = {len(sample_submission)}")

train の件数 = 699635
test の件数 = 299844
sample_submission の件数 = 299844


In [79]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   6996

## 各列について

参考 : https://www.kaggle.com/competitions/playground-series-s6e10/data?select=train.csv

- `id`
- `Age`
- `Flight Distance`
- `Departure Delay in Minutes` : 出発遅延時間（分）
- `Arrival Delay in Minutes` : 到着遅延時間（分）
- `Gender` : Male 50% / Female 50%
- `Customer Type` : Loyal Customer 82% / disloyal Customer 18%
- `Type of Travel` : Business travel 71% / Personal Travel 29%
- `Class` : Business 49% / Eco 47% / Other (30019) 4%
- `satisfaction` : true 44% / false 56%

### サービス系指標 : 以下は全て 0 ～ 5 の 6 段階評価

- `Inflight wifi service`
- `Departure/Arrival time convenient`
- `Ease of Online booking`
- `Gate location`
- `Food and drink`
- `Online boarding`
- `Seat comfort`
- `Inflight entertainment`
- `On-board service` : 機内サービス
- `Leg room service`
- `Baggage handling` : 手荷物取り扱い
- `Checkin service`
- `Cleanliness`

# 関数の定義

In [80]:
def train_lgb(train_df, test_df, feature_cols, target_col="satisfaction"):
    """
    指定された特徴量で LightGBM の交差検証学習を行い、
    OOF予測値（確率）、テスト予測値（確率）、学習済みモデル群を返す関数
    """
    
    # bool型をそのまま int（1/0）に変換
    y_train = train_df[target_col].astype(int).values
    
    # 【高速化①】ループの前にカテゴリ型への一括変換を済ませる
    X_train_df = train_df[feature_cols].copy()
    X_test_df = test_df[feature_cols].copy()
    
    for col in feature_cols:
        if X_train_df[col].dtype == 'object':
            X_train_df[col] = X_train_df[col].astype('category')
            X_test_df[col] = X_test_df[col].astype('category')

    # 【高速化②】.iloc のオーバーヘッドを避けるため、一回 pandas のままではなく 
    # LightGBM が最も得意とする DataFrame/NumPy の構造を維持する
    X_train_df = X_train_df.reset_index(drop=True)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',

        # 'colsample_bytree': 0.8,
        # 'subsample': 0.8,
        # 'subsample_freq': 1, # 💡 subsampleを動かす時は、1イテレーションごとにサンプリングし直すこの指定をセットで入れる
        
        'n_jobs': -1,  # 全コア並列使用
        'random_state': 0,
        'verbose': -1,
        'enable_categorical': True,
    }

    models = []
    oof_train = np.zeros(len(X_train_df))
    
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train_df, y_train)):
        # 【高速化③】iloc のスライスを最小限に
        X_tr = X_train_df.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_va = X_train_df.iloc[valid_idx]
        y_va = y_train[valid_idx]

        # Datasetの作成 (free_raw_data=True でメモリを節約)
        lgb_train = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
        lgb_eval = lgb.Dataset(X_va, y_va, reference=lgb_train, free_raw_data=True)

        # モデルの学習
        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_eval], # lgb_train を削除して検証データのみにする
            num_boost_round=1000,  # デフォルトは 1000 あたり

            # stopping_rounds のデフォルトは 100 あたり
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )

        best_iter = model.best_iteration
        best_score = model.best_score['valid_0']['auc']
        print(f"Fold {fold+1} | Best Iteration: {best_iter:4d} | Best AUC: {best_score:.5f}")

        oof_train[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
        
        models.append(model)

    # 全体の OOF ROC-AUC スコアを計算
    score = roc_auc_score(y_train, oof_train)
    print(f"\nOOF ROC-AUC Score: {score:.5f}")

    # 💡 すべての学習が終わった後、ループの外でテストデータを一括予測する
    # print("\nLoop finished. Predicting test data...")
    y_preds = [model.predict(X_test_df, num_iteration=model.best_iteration) for model in models]
    y_preds_mean = np.mean(y_preds, axis=0)

    return oof_train, y_preds_mean, models

# 周波数エンコーディング (Frequency Encoding)

- カテゴリの出現頻度を数値化できる

- 高カーディナリティ（種類が多い）カテゴリに強い

- ツリーモデルの分岐効率が上がる

In [81]:
# 出現頻度の「癖」が出やすい主要な変数を指定
freq_cols = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
]

# train と test を結合して全データにおける出現頻度（カウント）を計算
for col in freq_cols:
    if col in train.columns:
        # train と test の値を合わせてカウント
        full_series = pd.concat([train[col], test[col]])
        freq_map = full_series.value_counts().to_dict()

        # 新しい特徴量として追加（例: Age_freq）
        train[f"{col}_freq"] = train[col].map(freq_map)
        test[f"{col}_freq"] = test[col].map(freq_map)

print(
    f"周波数エンコーディング完了: {len(freq_cols)} 個の特徴量を生成しました。"
)

周波数エンコーディング完了: 4 個の特徴量を生成しました。


# EDA

## `travel_customer_type`

「旅行目的（Type of Travel）」×「ロイヤリティ」のクロス特徴量

In [82]:
train['travel_customer_type'] = train['Type of Travel'] + "_" + train['Customer Type']
test['travel_customer_type'] = test['Type of Travel'] + "_" + test['Customer Type']

## 不採用案

### サービス系指標の平均値

中間変数としては有効かもしれない。

In [83]:
# サービス系指標のリスト
service_cols = [
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness'
]

# 1. サービス全体の平均点
train['service_mean'] = train[service_cols].mean(axis=1)
test['service_mean'] = test[service_cols].mean(axis=1)

# 2. WiFi と他の主要サービスのギャップ（例: エンタメ - WiFi）
# train['wifi_entertainment_diff'] = train['Inflight entertainment'] - train['Inflight wifi service']
# test['wifi_entertainment_diff'] = test['Inflight entertainment'] - test['Inflight wifi service']

### 遅延フラグ

周波数エンコーディングとスコアが同等。

ならば、周波数エンコーディングの方が情報量が多いので、そっちを採用すべき。

In [84]:
# # 遅延フラグの作成
# delay_mapping = {
#     "Departure Delay in Minutes": "Departure_is_delayed",
#     "Arrival Delay in Minutes": "Arrival_is_delayed",
# }

# for orig_col, flag_col in delay_mapping.items():
    
#     # 0分より大きければ 1、そうでなければ 0（欠損値対策として fillna(0) を適用）
#     train[flag_col] = (train[orig_col].fillna(0) > 0).astype(int)
#     test[flag_col] = (test[orig_col].fillna(0) > 0).astype(int)

# print("遅延フラグの作成が完了しました。")

# 実行

In [85]:
%%time

feature_cols = [
    'Age',
    'Flight Distance',
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness',
    'Departure Delay in Minutes',
    'Arrival Delay in Minutes',
    'Gender',
    'Customer Type',
    'Type of Travel',
    'Class',
    'Age_freq',
    'Flight Distance_freq',
    'Departure Delay in Minutes_freq',
    'Arrival Delay in Minutes_freq',
    # 'service_mean',
    'travel_customer_type',
]


# 関数の実行
oof, preds, models = train_lgb(train, test, feature_cols)


# 特徴量重要度の可視化（表形式で出力）
feature_importances = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0,
)

importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": feature_importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(30))

Fold 1 | Best Iteration:  338 | Best AUC: 0.95895
Fold 2 | Best Iteration:  419 | Best AUC: 0.95776
Fold 3 | Best Iteration:  617 | Best AUC: 0.96036
Fold 4 | Best Iteration:  389 | Best AUC: 0.95848
Fold 5 | Best Iteration:  642 | Best AUC: 0.96046

OOF ROC-AUC Score: 0.95920


,feature,importance
0,Online boarding,1.412516e+06
1,Inflight wifi service,3.661724e+05
2,travel_customer_type,2.064011e+05
3,Type of Travel,2.014847e+05
4,Inflight entertainment,1.364012e+05
5,Checkin service,6.492708e+04
6,Baggage handling,6.326450e+04
7,Class,5.497969e+04
8,Seat comfort,5.124236e+04
9,On-board service,4.092919e+04


CPU times: user 6min 56s, sys: 1.2 s, total: 6min 58s
Wall time: 1min 48s


Fold 1 | Best Iteration:  338 | Best AUC: 0.95895

Fold 2 | Best Iteration:  419 | Best AUC: 0.95776

Fold 3 | Best Iteration:  617 | Best AUC: 0.96036

Fold 4 | Best Iteration:  389 | Best AUC: 0.95848

Fold 5 | Best Iteration:  642 | Best AUC: 0.96046

OOF ROC-AUC Score: 0.95920

# エラー分析

In [86]:
# 例: 実際は True (1) なのに、低い確率（0.1以下）と予測して外したデータを抽出
train['oof_pred'] = oof
error_cases = train[(train['satisfaction'] == 1) & (train['oof_pred'] < 0.1)]
print(f"大きく外した件数: {len(error_cases)}")
display(error_cases.head(10))

大きく外した件数: 12712


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Type of Travel,Class,satisfaction,Age_freq,Flight Distance_freq,Departure Delay in Minutes_freq,Arrival Delay in Minutes_freq,travel_customer_type,service_mean,oof_pred
77,77,27,1671,2,3,2,3,5,2,5,...,Personal Travel,Eco,True,25258,2266,907540,914855.0,Personal Travel_Loyal Customer,3.307692,0.039110
128,128,15,590,2,4,2,2,2,2,2,...,Personal Travel,Eco,True,6500,4520,907540,914855.0,Personal Travel_Loyal Customer,2.846154,0.033959
150,150,42,651,2,3,3,3,4,2,3,...,Business travel,Business,True,24916,4158,907540,914855.0,Business travel_Loyal Customer,2.384615,0.056029
160,160,31,1235,3,2,2,2,3,3,3,...,Business travel,Business,True,16222,4364,907540,914855.0,Business travel_Loyal Customer,2.846154,0.066875
206,206,56,694,3,5,3,2,2,5,4,...,Personal Travel,Eco,True,18397,674,907540,914855.0,Personal Travel_Loyal Customer,2.923077,0.078287
243,243,8,965,1,5,1,3,4,1,3,...,Personal Travel,Eco Plus,True,4289,1679,907540,914855.0,Personal Travel_Loyal Customer,3.461538,0.013606
257,257,52,577,2,2,2,2,1,4,3,...,Business travel,Business,True,22133,5057,907540,914855.0,Business travel_Loyal Customer,2.307692,0.070751
389,389,52,631,2,4,3,4,2,2,2,...,Business travel,Eco,True,22133,4355,907540,914855.0,Business travel_Loyal Customer,2.615385,0.069642
408,408,22,479,2,3,2,4,1,2,1,...,Personal Travel,Eco,True,23797,1484,907540,914855.0,Personal Travel_Loyal Customer,2.153846,0.029384
442,442,37,456,2,2,2,2,3,2,3,...,Business travel,Business,True,21475,883,907540,914855.0,Business travel_disloyal Customer,3.000000,0.069126
